# EO-LINCS project: Cube generation for Scientific Case Study (SCS) 1

### Explanatory power of novel EO data streams for predicting net carbon fluxes

**Objective**: The SCS1 trains an artificial neural network (ANN) to predict carbon fluxes, where meteorological and reflectance data from satellites are taken as input. The training will be based on the in-situ observations from eddy covariance flux tower provided by the FLUXNET2015 dataset.

**Outcomes**: A working data processing chain to incorporate Sentinel-2 data into the FLUXCOM-X framework 
that is updatable and expandable to all sites and other Sentinel data products. An analysis of the contributions 
of Sentinel-2 data for predicting NEE and analysis into the added value with regards to interannual variability, 
drought responses, and disturbance.

**Required datasets**:
* [Sentinel-2 L2A from CDSE](https://browser.stac.dataspace.copernicus.eu/?.language=en)
* [ERA-5 land](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land?tab=overview)
* [ESA CCI Biomass](https://climate.esa.int/en/projects/biomass/)


-------------------------------------------------------------

The following notebook shows how the users can load data from various sources defined in `scs1_config.yml` using the `MultiSourceDataStore` tool.

#### **What You Can Do with This Notebook** 
- Generate a configuration file based on fluxtower locations
- Load datasets from various sources as defined in the generated `scs1_config.yml`
- View the progress of each data request to the `MultiSourceDataStore`
- Quickly preview the datasets by plotting them.

#### **Requirements**  
Before proceeding, ensure you have the necessary dependencies installed (which is just one in this case!).
- `xcube-multistore`

It is available via conda-forge, and you can install it using
`conda install --channel conda-forge xcube-multistore`

Once you have it installed, you are ready to proceed.

This Multistore mainly works with a file called `scs1_config.yml` which is at the same file level as this notebook.
To understand what goes into the schema, you can read more [here](https://xcube-dev.github.io/xcube-multistore/).

-------------------------------------------------------

Let's import the `MultiSourceDataStore`

In [1]:
import yaml

import pandas as pd
from xcube_multistore.multistore import MultiSourceDataStore
from xcube_multistore.utils import get_bbox

You can find out how to fill out the config file by also using this super helpful function `get_config_schema()`. Run it and try expand the fields to learn more about the possible properties that the configuration file accepts along with the [Configuration Guide](https://xcube-dev.github.io/xcube-multistore/config/).

In [2]:
MultiSourceDataStore.get_config_schema()

This science case requires data from several flux sites. So we define them in a `scs1_sites.csv` file for easier management and access. For the purpose of this example, we will focus on the first 3 sites

In [3]:
sites = pd.read_csv("scs1_sites.csv")
sites = sites.iloc[:3]
sites

,Site ID,latitude,longitude,IGBP
0,AU-Dry,-15.2588,132.3706,SAV
1,AU-How,-12.4943,131.1523,WSA
2,BE-Lon,50.5516,4.7462,CRO


In the following cell, we will create the config object which will then be saved as `scs1_config.yml` for persistance and ready to be read by `MultiSourceDataStore`.

To read more about how this config file is structured, you can find the [Configuration Guide here](https://xcube-dev.github.io/xcube-multistore/config/).

Specifically, we are using the [single dataset object](https://xcube-dev.github.io/xcube-multistore/config/#single-dataset-object) and [data stores](https://xcube-dev.github.io/xcube-multistore/config/#store-object) schemas here.

In [4]:
config = dict(datasets=[])
for index, site in sites.iterrows():
    bbox_final, crs_final = get_bbox(
        site["latitude"], site["longitude"], cube_width=4000, crs_final="utm"
    )

    # append config for Sentinel-2
    config_ds = dict(
        identifier=f"{site['Site ID']}_sen2",
        store="stac-cdse",
        data_id="sentinel-2-l2a",
        open_params=dict(
            time_range=["2019-03-01", "2019-03-15"],
            bbox=bbox_final,
            spatial_res=10,
            crs=f"EPSG:{crs_final.to_epsg()}",
            apply_scaling=True,
            asset_names=[
                "B01",
                "B02",
                "B03",
                "B04",
                "B05",
                "B06",
                "B07",
                "B08",
                "B8A",
                "B09",
                "B11",
                "B12",
                "SCL",
            ],
        ),
    )
    config["datasets"].append(config_ds)

    # append config for ERA5
    config_ds = dict(
        identifier=f"{site['Site ID']}_era5land",
        store="cds",
        data_id="reanalysis-era5-land",
        open_params=dict(
            variable_names=["2m_temperature", "total_precipitation"],
            time_range=["2019-03-01", "2019-03-15"],
            point=[site["latitude"], site["longitude"]],
            spatial_res=0.1,
        ),
    )
    config["datasets"].append(config_ds)

    # append config for ESA CCI
    config_ds = dict(
        identifier=f"{site['Site ID']}_ccibiomass",
        store="esa_cci",
        grid_mapping=f"{site['Site ID']}_sen2",
        data_id="esacci.BIOMASS.yr.L4.AGB.multi-sensor.multi-platform.MERGED.5-0.100m",
        open_params=dict(
            time_range=["2019-01-01", "2019-12-31"],
        ),
    )
    config["datasets"].append(config_ds)

# define stores
config["data_stores"] = []
# add storage data store
config_store = dict(
    identifier="storage",
    store_id="file",
    store_params=dict(root="data"),
)
config["data_stores"].append(config_store)
# add ESA CCI data store
config_store = dict(
    identifier="esa_cci",
    store_id="cciodp",
)
config["data_stores"].append(config_store)
# add STAC data store
config_store = dict(
    identifier="stac-cdse",
    store_id="stac-cdse",
    store_params=dict(
        key="OX94JKCFEF1PLNVNAU2J",
        secret="ceucxUr3s8yNhaP8k3g0FnQERKgm8SUfd9TdSFrF",
        stack_mode=True,
    ),
)
config["data_stores"].append(config_store)
# add CDS data store
config_store = dict(
    identifier="cds",
    store_id="cds",
    store_params=dict(
        endpoint_url="https://cds.climate.copernicus.eu/api",
        cds_api_key="8ba066d4-5195-4c1d-af91-6fb13bfc22df",
        normalize_names=True,
    ),
)
config["data_stores"].append(config_store)

In [5]:
with open("scs1_config.yml", "w") as file:
    yaml.dump(config, file, sort_keys=False)

Now, we can initialize the `MultiSourceDataStore` by passing the path to the `scs1_config.yml` which currently is on the same level as this notebook.

By running the cell below, you would start seeing progress tables for each data that you requested in the `scs1_config.yml`.

In [ ]:
msds = MultiSourceDataStore("scs1_config.yml")

<frozen abc>:106: FutureWarning: xarray subclass VectorDataCube should explicitly define __slots__


Dataset identifier,Status,Message,Exception
AU-Dry_sen2,STARTED,Write dataset 'AU-Dry_sen2'.,-
AU-Dry_era5land,WAITING,-,-
AU-Dry_ccibiomass,WAITING,-,-
AU-How_sen2,WAITING,-,-
AU-How_era5land,WAITING,-,-
AU-How_ccibiomass,WAITING,-,-
BE-Lon_sen2,WAITING,-,-
BE-Lon_era5land,WAITING,-,-
BE-Lon_ccibiomass,WAITING,-,-


/home/konstantin/micromamba/envs/xcube-multistore/lib/python3.12/site-packages/numpy/_core/numeric.py:366: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(a, fill_value, casting='unsafe')


We can now open the data using the xcube datastore framework API as usual. Note that the multi-source data store requires a data store called `storage`, which is configured in our `scs1_config.yml` under the `data_stores` section.

In [ ]:
ds = msds.stores.storage.open_data("AU-Dry_sen2.zarr")
ds

We can now select a variable for one timestep and plot it for a quick preview of the data

In [ ]:
ds.B04.isel(time=0).plot(vmin=0., vmax=0.2)

In [ ]:
ds = msds.stores.storage.open_data("AU-Dry_era5land.zarr")
ds

In [ ]:
ds.t2m.plot()

In [ ]:
ds = msds.stores.storage.open_data("AU-Dry_ccibiomass.zarr")
ds

In [ ]:
ds["agb"].plot()